# Building a workflow node-by-node (server-first)

Earlier notebooks built a workflow in **one shot**: you assemble a
`WorkflowConfigFullyHydrated` (nodes + edges) entirely on the client and hand the whole
thing to `client.workflows.create_from_config(...)`, which persists it in a single call.

This notebook shows the **other way** — build the graph **incrementally on the server**,
one entity at a time:

1. Create an empty **workflow** (`client.workflows.create`)
2. Create each **node** as its own server-side record (`client.nodes.create`), attached
   to the workflow via `workflow_id`
3. Connect them by creating **edges** (`client.edges.create`)
4. Publish and activate a **version**
5. Update a node **in place** and publish another version
6. Fetch the server-side records back by id, then **activate** the new version
7. **Clone** the workflow

Each node and edge is an addressable record with its own `id` and `logical_id`, so you can
create, fetch, update, and delete them independently — handy when a UI (or a script) edits a
graph piece by piece rather than re-uploading the whole config. This is the same model the
workflow editor uses under the hood.

> **Tip:** Node and edge configs are still the typed classes from `interactly.configs`
> (`pip install "interactly[configs]"`); here we save each one individually instead of nesting
> them in a `WorkflowConfigFullyHydrated`. See [`17_interactly_configs.ipynb`](17_interactly_configs.ipynb)
> for a deep dive on the typed config classes.

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## 1. Create a workflow

In [ ]:
from interactly.types.workflows.workflow import Workflow

# Start with an EMPTY workflow — no nodes or edges yet. We'll attach them one by
# one in the next sections (contrast with create_from_config, which uploads the
# whole graph at once).
workflow: Workflow = await client.workflows.create(
    name="Customer Support Agent",
    description="Built node-by-node in 16_nodes_and_edges.ipynb",
)

WF_ID = workflow.id
print(f"Workflow id={WF_ID}")

## 2. Create nodes

Nodes are independent entities keyed by a `logical_id`. Build each one with a typed
config class and attach it to the workflow via `workflow_id`. Inspect the JSON Schema
for any node type with `client.nodes.schema(<node_type>)`.

In [ ]:
from typing import List, Dict, Any

# List all available node types and their categories
node_types: List[Dict[str, Any]] = await client.nodes.types()
print("Available Node Types:")
for nt in node_types:
    print(f"  - {nt.get('type')} ({nt.get('primary_category')} / {nt.get('secondary_category')})")

In [ ]:
from typing import Dict, Any
from interactly.configs import get_node_config_class

# Inspect the schema for a node type (requires the node_type argument)
schema: Dict[str, Any] = await client.nodes.schema("say_llm")

# Discover the associated config class programmatically
config_class = get_node_config_class("say_llm")
class_name = config_class.__name__ if config_class else "Unknown"

print(f"say_llm config class: interactly.configs.{class_name}")
print("say_llm config fields:", list(schema.get("config_schema", {}).get("properties", {}).keys())[:5], "…")

In [ ]:
from interactly.types.nodes.node import Node

from interactly.configs import SayLLMNodeConfig, PromptConfig

# Create a greeting LLM node (the start node)
greeting_node: Node = await client.nodes.create(
    node_config=SayLLMNodeConfig(
        workflow_id=WF_ID,
        name="Greet User",
        is_start=True,
        main_response_config=PromptConfig(
            prompt="You are a friendly support agent. Greet the user warmly and ask how you can help.",
        ),
    )
)

GREETING_NODE_ID = greeting_node.node_config.logical_id
print(f"Greeting node  id={greeting_node.id}  logical_id={GREETING_NODE_ID}")

In [ ]:
from interactly.types.nodes.node import Node

from interactly.configs import SayLLMNodeConfig, PromptConfig

# Create a farewell / end-conversation node
farewell_node: Node = await client.nodes.create(
    node_config=SayLLMNodeConfig(
        workflow_id=WF_ID,
        name="Farewell User",
        main_response_config=PromptConfig(
            prompt="You are a friendly support agent. Say goodbye to the user warmly and thank them for their time.",
        ),
    )
)

FAREWELL_NODE_ID = farewell_node.node_config.logical_id
print(f"Farewell node  id={farewell_node.id}  logical_id={FAREWELL_NODE_ID}")

## 3. Connect nodes with edges

In [ ]:
from interactly.types.edges.edge import Edge

from interactly.configs import DirectEdgeConfig

edge: Edge = await client.edges.create(
    edge_config=DirectEdgeConfig(
        workflow_id=WF_ID,
        name="Greeting → Farewell",
        source_node_logical_id=GREETING_NODE_ID,
        destination_node_logical_id=FAREWELL_NODE_ID,
    )
)

EDGE_ID = edge.edge_config.logical_id
print(f"Edge  id={edge.id}  {GREETING_NODE_ID} → {FAREWELL_NODE_ID}")

## 4. Publish a version and activate it

In [ ]:
from interactly.types.workflows.workflow import WorkflowVersion

v1: WorkflowVersion = await client.workflows.versions.create(
    WF_ID,
    version_name="v1 — initial",
    mark_as_active=True,
)

V1_NUMBER = v1.version_number
print(f"Published version  number={V1_NUMBER}  active={v1.is_active}")

## 5. Update a node in place, then publish a new version

Because each node is its own server record, you can change **one** without touching the
rest of the graph. `client.nodes.update()` **PATCH-merges** the fields you pass (here just
the prompt), leaving `name`, `is_start`, etc. intact. We re-fetch the node to confirm, then
publish a fresh version.

In [ ]:
from interactly.configs import SayLLMNodeConfig, PromptConfig
from interactly.types.workflows.workflow import WorkflowVersion

# PATCH-merge just the prompt on the greeting node (identified by its server id).
await client.nodes.update(
    node_id=greeting_node.id,
    node_config=SayLLMNodeConfig(
        main_response_config=PromptConfig(
            prompt="You are an expert support agent. Greet warmly and collect the issue details.",
        ),
    ),
)

# Re-fetch that single node to confirm the patch stuck — and that name/is_start survived.
refreshed = await client.nodes.get(greeting_node.id)
print("greeting prompt now:", refreshed.node_config.main_response_config.prompt)
print("name:", refreshed.node_config.name, " is_start:", refreshed.node_config.is_start)

v2: WorkflowVersion = await client.workflows.versions.create(
    WF_ID,
    version_name="v2 — refined greeting prompt",
)
V2_NUMBER = v2.version_number
print(f"\nPublished version number={V2_NUMBER}")

## 6. Fetch the records back by id

Each node and edge we created is an addressable server record. Fetch any of them back on
demand with `client.nodes.get(id)` / `client.edges.get(id)` — the same handles a UI would
use to load one piece of a graph without pulling the whole workflow.

In [ ]:
# Each node and edge we created is an addressable server record — fetch any of them
# straight back by its `id` with client.nodes.get() / client.edges.get().
for nid in (greeting_node.id, farewell_node.id):
    n = await client.nodes.get(nid)
    cfg = n.node_config
    print(f"node  {n.id}  {cfg.name!r}  type={cfg.type}  is_start={cfg.is_start}")

e = await client.edges.get(edge.id)
ec = e.edge_config
print(f"edge  {e.id}  {ec.source_node_logical_id} → {ec.destination_node_logical_id}")

## 7. Activate v2

In [ ]:
from typing import List

from interactly.types.workflows.workflow import Workflow, WorkflowVersion

# activate() returns the updated workflow, whose active_version_number is the
# authoritative pointer to the live version (the per-version `is_active` field is
# not populated by the list endpoint — the workflow owns the "which is active" fact).
updated: Workflow = await client.workflows.versions.activate(WF_ID, V2_NUMBER)
active_number = updated.active_version_number
print(f"Active version is now v{active_number}")

# versions.list returns a plain list of WorkflowVersion objects; compare each
# version's number against the workflow's active_version_number to flag the live one.
versions: List[WorkflowVersion] = await client.workflows.versions.list(WF_ID)
for v in versions:
    marker = "  <- active" if v.version_number == active_number else ""
    print(f"  v{v.version_number}  {v.version_name!r}{marker}")

## 8. Clone the workflow

In [ ]:
from interactly.types.workflows.workflow import Workflow

clone: Workflow = await client.workflows.clone(WF_ID, name="Customer Support Agent — Copy")
print(f"Cloned workflow  id={clone.id}  name={clone.name!r}")

CLONE_ID = clone.id

## 8b. Three configs that change *when* a node or edge runs

Everything above is about *where* execution goes. These change *when* it happens — each has its
own notebook, but they belong in your mental model of the building blocks.

| Config | Goes on | Effect |
|---|---|---|
| `SelfLoopConfig` | a **node** | Bounded re-execution — [notebook 20](20_waiting_conditions.ipynb) context, [self-loops guide](../docs/guides/self_loops.md) |
| `CompanionThreadConfig` | a **direct edge** | Forks a background thread — [notebook 19](19_companion_threads.ipynb) |
| `EvaluateWhileWaitingConfig` | a **conditional edge** | Fires while the source node is parked — [notebook 20](20_waiting_conditions.ipynb) |

In [ ]:
import interactly_configs as ic

# A bound is REQUIRED — a loop that could never terminate fails at construction, client-side,
# before any request is sent.
try:
    ic.SelfLoopConfig(enabled=True)
except Exception as exc:
    print(f"✅ unbounded self-loop refused: {type(exc).__name__}")

bounded = ic.SelfLoopConfig(enabled=True, max_retries=3, expiry_time=30, time_between_retries=2)
print(f"max_retries counts RETRIES, not attempts: {bounded.max_retries} → "
      f"up to {bounded.max_retries + 1} executions")

### The no-op node

`no_op` runs and succeeds without doing anything else. Three uses: a **fan-in junction** so several
branches converge on one outgoing edge, a **placeholder** for an unbuilt step, and a deterministic
**stand-in** in tests.

**It is not `disabled=True`.** A disabled node is skipped entirely and emits no events, so
downstream conditional edges have nothing to branch on. A no-op node *runs*: it emits the usual
node start/end events and writes `<name>` and `<name>_success` to thread state — which is exactly
what makes it usable as a junction.

In [ ]:
junction = ic.NoOpNodeConfig(name="Junction", output_runtime_variable_name="junction")
placeholder = ic.NoOpNodeConfig(name="Eligibility check", note="TODO: replace with the real check")

# Testing-only knobs: simulate a slow node, or a failure branch with no failing integration.
slow = ic.NoOpNodeConfig(name="Slow", delay_seconds=2.0)
failing = ic.NoOpNodeConfig(name="Boom", simulate_failure=True)

print(f"type              : {junction.type}")
print(f"writes            : {junction.output_runtime_variable_name} / "
      f"{junction.output_runtime_variable_name}_success")
print(f"simulate_failure  : reports success=False without aborting the run "
      f"({failing.simulate_failure})")

### Reading edge configs back

Four defensive accessors read the nested configs safely on **any** edge, including edges stored
before these features existed. Use them rather than reaching into the nested objects yourself.

In [ ]:
poller_id = "node_example"
companion_edge = ic.DirectEdgeConfig(
    source_node_logical_id="node_a", destination_node_logical_id=poller_id,
    companion_thread_config=ic.CompanionThreadConfig(is_companion_thread=True, thread_id="bg"),
)
plain_edge = ic.DirectEdgeConfig(source_node_logical_id="node_a", destination_node_logical_id="node_b")

for e in (companion_edge, plain_edge):
    print(f"companion={ic.edge_is_companion(e)!s:5s}  "
          f"thread_id={ic.edge_companion_thread_id(e)!s:6s}  "
          f"waits={ic.edge_evaluates_while_waiting(e)!s:5s}  "
          f"waiting_cfg={ic.edge_waiting_evaluation_config(e)}")

## 9. Cleanup

In [ ]:
await client.workflows.delete(CLONE_ID)
await client.workflows.delete(WF_ID)
print("Both workflows deleted.")